In [0]:
"""
05_packaging_kpis.py

Streaming Packaging KPIs.

Computes packaging KPIs from Silver Packaging Events.

Input:
    packaging_events

Output:
    packaging_kpis

Author:
Sumanth Vempalle

Version:
2.1.0
"""

import dlt

from pyspark.sql.functions import (
    avg,
    col,
    count,
    current_timestamp,
    max,
    min,
    sum,
    when,
)


# ============================================================
# Packaging KPIs
# ============================================================

@dlt.table(
    name="packaging_kpis",
    comment="Packaging and shipment readiness KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dlt.expect_or_drop(
    "valid_product_code",
    "product_code IS NOT NULL",
)

@dlt.expect_or_drop(
    "valid_package_type",
    "package_type IS NOT NULL",
)

@dlt.expect(
    "positive_packages_completed",
    "packages_completed > 0",
)

def packaging_kpis():

    packaging = (
         dlt.read_stream(
            "packaging_events"
    )
    
    .withWatermark(
        "event_timestamp",
        "10 minutes",
    )

)
    return (

        packaging

        .groupBy(

            "plant_code",

            "product_code",

            "product_name",

            "family",

            "package_type",

        )

        .agg(

            count("*").alias(
                "packages_completed"
            ),

            avg(
                "package_weight_kg"
            ).alias(
                "average_package_weight_kg"
            ),

            min(
                "package_weight_kg"
            ).alias(
                "minimum_package_weight_kg"
            ),

            max(
                "package_weight_kg"
            ).alias(
                "maximum_package_weight_kg"
            ),

            sum(

                when(
                    col("packaging_status") == "READY_FOR_SHIPMENT",
                    1,
                ).otherwise(0)

            ).alias(
                "ready_for_shipment"
            ),

        )

        .withColumn(

            "shipment_readiness_rate",

            (
                col("ready_for_shipment")
                /
                col("packages_completed")
            ) * 100

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )